## UFO Sightings

In [11]:
# %pip install altair

In [1]:
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [18]:
df = pd.read_csv("https://uiuc-ischool-dataviz.github.io/spring2019online/week04/data/ufo-scrubbed-geocoded-time-standardized-00.csv",
                  names = ["date", "city", "state", "country", "shape", "duration_seconds", "duration", "comment", "report_date", "latitude", "longitude"],
                  parse_dates = ["date", "report_date"])

In [ ]:
df = df.dropna(subset=["date", "shape"])
df["year"] = df["date"].dt.year
df = df[df["year"] >= 1990]
top_shapes = df["shape"].value_counts().head(8).index
df = df[df["shape"].isin(top_shapes)]
df.head()

,date,city,state,country,shape,duration_seconds,duration,comment,report_date,latitude,longitude,year
48,1990-10-10 21:00:00,ashburn,ga,us,triangle,120.0,2 minutes,Translucent Craft that makes No Sound While Mo...,2007-10-08,31.705833,-83.653333,1990
49,1991-10-10 22:00:00,frisco,nc,us,unknown,1800.0,30 minutes,A friend and myself were standing on the shore...,1999-01-28,35.235000,-75.628889,1991
50,1991-10-10 22:00:00,harrisburg,pa,us,triangle,600.0,10 minutes,We observed 3 triangular shaped high speed obj...,2003-05-09,40.273611,-76.884722,1991
52,1992-10-10 18:00:00,stafford,tx,us,unknown,10.0,10 seconds,A man emerges from a beam of light that shines...,2012-04-18,29.615833,-95.557500,1992
54,1992-10-10 22:00:00,weatherford,tx,us,unknown,30.0,30 seconds,Black or dark grey. Too big&#44 too low&#44 t...,2005-09-02,32.759167,-97.796944,1992


In [21]:
yearly = df.groupby(["year", "shape"]).size().reset_index(name="count")

brush = alt.selection_interval(encodings=["x"], value={"x": [2000, 2010]})
shape_sel = alt.selection_point(fields=["shape"], bind="legend")

In [22]:
timeline = alt.Chart(yearly).mark_area().encode(
    x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d")),
    y=alt.Y("count:Q", title="Number of Sightings"),
    color=alt.Color("shape:N", scale=alt.Scale(scheme="tableau10"), title="Shape"),
    opacity=alt.condition(shape_sel, alt.value(1), alt.value(0.2)),
    tooltip=["year:Q", "shape:N", "count:Q"]
).add_params(
    brush, shape_sel
).properties(
    width=600, height=250
)

In [23]:
bars = alt.Chart(yearly).mark_bar().encode(
    x=alt.X("sum(count):Q", title="Sightings in Selected Years"),
    y=alt.Y("shape:N", sort="-x", title=None),
    color=alt.Color("shape:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["shape:N", "sum(count):Q"]
).transform_filter(
    brush
).properties(
    width=600, height=250
)

In [24]:
chart = timeline & bars
chart

alt.VConcatChart(...)

In [34]:
import os
os.makedirs("../assets/json", exist_ok=True)
chart.save("../assets/json/ufo_chart.json")
print(os.listdir("../assets/json"))

['ufo_chart.html', 'population_scatter_small.json', 'buildings_sqft.json', 'dashboard_mobility.json', 'altair_mobility_dashboard.json', 'cars.json', 'static_mobility.json', 'direct_from_editor.json', 'corgis_dotchart_world_smooth.json', 'chart1.json', 'population_scatter.json', 'corgis_dotchart_world.json', 'corgis_choro_world.json', 'buildings_sqft_stats.json', 'ufo_chart.json', 'altair_mobility_data_dashboard.json']
